In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
import lightgbm as lgb
from xgboost import XGBClassifier
from sklearn import svm
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_validate
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
#import joblib

datos = pd.read_csv('Gans/GAN_MIX.csv')

X = datos.drop(['PRED'], axis=1)
y = datos['PRED']


X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, test_size=0.2, random_state=42)

model = RandomForestClassifier(random_state=42).fit(X_train, y_train)
#model = MLPClassifier(random_state=42).fit(X_train, y_train)
#model = lgb.LGBMClassifier(random_state=42).fit(X_train, y_train)
#model = XGBClassifier(random_state=42).fit(X_train, y_train)
#model = svm.SVC(random_state=42, probability=True).fit(X_train, y_train)

#joblib.dump(model, 'best_model.pkl')

# Perform cross-validation
scoring = ['accuracy', 'f1_macro', 'precision_macro', 'recall_macro']
scores = cross_validate(model, X_train, y_train, scoring=scoring, cv=10, return_train_score=True)

# Extract the average scores
accuracy = scores['test_accuracy'].mean()
F1_score = scores['test_f1_macro'].mean()
precision = scores['test_precision_macro'].mean()
sensitivity_recall = scores['test_recall_macro'].mean()

print("Accuracy:", accuracy)
print("F1_score:", F1_score)
print("Precision:", precision)
print("Sensitivity/Recall:", sensitivity_recall)

print('\n')

# Realizar predicciones en los datos de prueba
y_pred = model.predict(X_test)

# Calcular las métricas en los datos de prueba
accuracy_test = accuracy_score(y_test, y_pred)
f1_score_test = f1_score(y_test, y_pred, average='macro')
precision_test = precision_score(y_test, y_pred, average='macro')
recall_test = recall_score(y_test, y_pred, average='macro')

# Imprimir las métricas en los datos de prueba
print("Accuracy (Testing):", accuracy_test)
print("F1_score (Testing):", f1_score_test)
print("Precision (Testing):", precision_test)
print("Sensitivity/Recall (Testing):", recall_test)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_predict 
from sklearn.metrics import roc_curve, auc

# Perform cross-validation and predict probabilities
y_probas_cv = cross_val_predict(model, X_train, y_train, cv=10, method="predict_proba")

# Compute ROC curve and AUC for each class
fpr_cv, tpr_cv, thresholds_cv = roc_curve(y_train, y_probas_cv[:, 1])
auc_cv = auc(fpr_cv, tpr_cv)

# Plot ROC curve for cross-validation
plt.figure(figsize=(8, 6))
plt.plot(fpr_cv, tpr_cv, label=f'ROC (AUC = {auc_cv:.2f})', color='blue')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) - Cross-validation')
plt.legend(loc='lower right')
plt.savefig('Gans/SVM_PAAC_training_mix', dpi=600)
plt.show()

# Realizar predicciones en los datos de prueba
y_probas_test = model.predict_proba(X_test)[:, 1]

# Compute ROC curve and AUC for test data
fpr_test, tpr_test, thresholds_test = roc_curve(y_test, y_probas_test)
auc_test = auc(fpr_test, tpr_test)

# Plot ROC curve for test data
plt.figure(figsize=(8, 6))
plt.plot(fpr_test, tpr_test, label=f'ROC (AUC = {auc_test:.2f})', color='red')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) - Testing')
plt.legend(loc='lower right')
plt.savefig('Gans/SVM_PAAC_testing_mix', dpi=600)
plt.show()